# Stage 8: Create LlamaIndex Documents

**Purpose**: Converts the processed JSON data into LlamaIndex `Document` objects for use in retrieval experiments.

**Input**: `data/intermediate/OBRA CIVIL/OBRA CIVIL_stage7.json`  
**Output**: 
- `data/processed/OBRA CIVIL_texto.pkl` (target documents)
- `data/processed/OBRA CIVIL_resumen.pkl` (query documents)

## What this notebook does

Creates two parallel document collections:

### Resumen documents (queries)
```python
Document(
    text="Canalización hormigonada de 2 tubos...",
    metadata={
        "item_key": "OEB020$AAAAAA",
        "type": "resumen",
        "parent_key": "OEB020$",
        "ud": "m",
        "concept": "...",
        "parameters": {...}
    }
)
```

### Texto documents (retrieval targets)
Same structure with `type: "texto"` and full description text.

### Why LlamaIndex format?

LlamaIndex `Document` objects provide:
- Standardized interface for various retrieval backends (BM25, dense, hybrid)
- Metadata preservation for evaluation (item_key matching, parent-level grouping)
- Serialization via pickle for efficient loading

In [10]:
!pip install -q llama_index

In [11]:
from pathlib import Path
import json
import pickle
from llama_index.core import Document
from utils import config

def main(file):
    
    # Load the JSON data

    json_file_path = config.stage_path(file, 7)
    save_path_texto = config.PROCESSED_DIR / f"{file}_texto.pkl"
    save_path_resumen = config.PROCESSED_DIR / f"{file}_resumen.pkl"

    with open(json_file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)

    # Create a list of documents
    documents_texto = []
    documents_resumen = []

    # Iterate over each item in the JSON data
    for item_key, item_value in data.items():
        # Extract contents
        texto = item_value['texto']
        resumen = item_value['resumen']
    
        # Extract metadata
        metadata = {
            "item_key": item_key,
            "ud": item_value["ud"],
            "concept": item_value["concept"],
            "parent_key": item_value["parent_key"],
            "parameters": item_value["parameters"]
        }
    
        # Create the 'resumen' document
        resumen_doc = Document(
            text=resumen,
            metadata={**metadata, "type": "resumen"}
        )
        
        documents_resumen.append(resumen_doc)
    
        # Create the 'texto' document
        texto_doc = Document(
            text=texto,
            metadata={**metadata, "type": "texto"}
        )
        
        documents_texto.append(texto_doc)

    # Save the documents to a file
    try:
        with open(save_path_texto, 'wb') as file:
            pickle.dump(documents_texto, file)
        print(f"Successfully saved {len(documents_texto)} documents to {save_path_texto}")
    except Exception as e:
        print(f"An error occurred while saving the documents: {str(e)}")

    try:
        with open(save_path_resumen, 'wb') as file:
            pickle.dump(documents_resumen, file)
        print(f"Successfully saved {len(documents_resumen)} documents to {save_path_resumen}")
    except Exception as e:
        print(f"An error occurred while saving the documents: {str(e)}")

    # Now 'documents' contains all the LlamaIndex documents and they are saved to a file

In [12]:
main('OBRA CIVIL')

Successfully saved 111644 documents to /work/data/processed/OBRA CIVIL_texto.pkl
Successfully saved 111644 documents to /work/data/processed/OBRA CIVIL_resumen.pkl
